## Stage3 Replay Preprocessing
Build replay-ready CSVs in `stage3_output` (separate from training notebook).

In [1]:
import pandas as pd
from pathlib import Path

STAGE2_TRAIN = Path(r"data\stage2\stage2_combined_train.csv")
STAGE2_VAL = Path(r"data\stage2\stage2_combined_val.csv")
STAGE2_TEST = Path(r"data\stage2\stage2_combined_test.csv")
STAGE3_MERGED = Path(r"stage3_output\stage3_merged.csv")
OUT = Path(r"stage3_output")
OUT.mkdir(parents=True, exist_ok=True)

UNIFIED_CLASSES = [
    'MEL','BCC','SCC','AK','NEV','BKL','DF','VASC','SEK',
    'TINEA','PSORIASIS','VITILIGO','MELASMA','FUNGAL','ECZEMA','URTICARIA'
]

RAW_TO_UNIFIED = {
    'mel':'MEL','nv':'NEV','bcc':'BCC','akiec':'AK','ak':'AK','ack':'AK','bkl':'BKL','df':'DF','vasc':'VASC','scc':'SCC','sek':'SEK',
    'MEL':'MEL','NV':'NEV','BCC':'BCC','AK':'AK','BKL':'BKL','SCC':'SCC','VASC':'VASC','DF':'DF'
}

def map_label(v):
    if pd.isna(v):
        return 'UNKNOWN'
    s = str(v).strip()
    if s in UNIFIED_CLASSES:
        return s
    return RAW_TO_UNIFIED.get(s.lower(), RAW_TO_UNIFIED.get(s, 'UNKNOWN'))

print('Config loaded.')

Config loaded.


In [2]:
s2_train = pd.read_csv(STAGE2_TRAIN)
s2_val = pd.read_csv(STAGE2_VAL)
s2_test = pd.read_csv(STAGE2_TEST)
s3 = pd.read_csv(STAGE3_MERGED)

def prep_stage2(df, split_name):
    out = df.copy()
    out['unified_label'] = out['label'].map(map_label)
    out = out[out['unified_label'] != 'UNKNOWN'].copy()
    out['dataset_source'] = out['source'].astype(str)
    out['fst_group'] = pd.to_numeric(out.get('fst_group', 0), errors='coerce').fillna(0).astype(int)
    out['split'] = split_name
    out['has_disease_label'] = True
    return out[['image_path','unified_label','fst_group','split','dataset_source','has_disease_label']]

def prep_stage3(df):
    out = df.copy()
    out['unified_label'] = out['unified_label'].map(map_label)
    out = out[out['unified_label'].isin(UNIFIED_CLASSES)].copy()
    out['fst_group'] = pd.to_numeric(out.get('fst_group', 0), errors='coerce').fillna(0).astype(int)
    out['split'] = out.get('split', 'train').astype(str).str.lower()
    out.loc[~out['split'].isin(['train','val','test']), 'split'] = 'train'
    out['has_disease_label'] = True
    return out[['image_path','unified_label','fst_group','split','dataset_source','has_disease_label']]

s2_train_p = prep_stage2(s2_train, 'train')
s2_val_p = prep_stage2(s2_val, 'val')
s2_test_p = prep_stage2(s2_test, 'test')
s3_p = prep_stage3(s3)

s3_train = s3_p[s3_p['split'] == 'train'].copy()
s3_val = s3_p[s3_p['split'] == 'val'].copy()
s3_test = s3_p[s3_p['split'] == 'test'].copy()

replay_train = pd.concat([s2_train_p, s3_train], ignore_index=True)
replay_val = pd.concat([s2_val_p, s3_val], ignore_index=True)
replay_test = pd.concat([s2_test_p, s3_test], ignore_index=True)

pad_train = s2_train_p[s2_train_p['dataset_source'].str.upper().eq('PAD-UFES-20')].copy()
pad_val = s2_val_p[s2_val_p['dataset_source'].str.upper().eq('PAD-UFES-20')].copy()
pad_test = s2_test_p[s2_test_p['dataset_source'].str.upper().eq('PAD-UFES-20')].copy()

In [3]:
replay_train.to_csv(OUT / 'stage3_replay_train.csv', index=False)
replay_val.to_csv(OUT / 'stage3_replay_val.csv', index=False)
replay_test.to_csv(OUT / 'stage3_replay_test.csv', index=False)
pad_train.to_csv(OUT / 'pad_monitor_train.csv', index=False)
pad_val.to_csv(OUT / 'pad_monitor_val.csv', index=False)
pad_test.to_csv(OUT / 'pad_monitor_test.csv', index=False)

print('Saved replay CSVs in stage3_output:')
print(' - stage3_replay_train.csv', len(replay_train))
print(' - stage3_replay_val.csv  ', len(replay_val))
print(' - stage3_replay_test.csv ', len(replay_test))
print(' - pad_monitor_val.csv    ', len(pad_val))

print('\nReplay train source mix:')
print(replay_train['dataset_source'].value_counts())

print('\nReplay val source mix:')
print(replay_val['dataset_source'].value_counts())

Saved replay CSVs in stage3_output:
 - stage3_replay_train.csv 27624
 - stage3_replay_val.csv   3170
 - stage3_replay_test.csv  6163
 - pad_monitor_val.csv     324

Replay train source mix:
dataset_source
ISIC2019          13255
dermnet           10390
fitzpatrick17k     2553
PAD-UFES-20        1426
Name: count, dtype: int64

Replay val source mix:
dataset_source
ISIC2019          2494
fitzpatrick17k     352
PAD-UFES-20        324
Name: count, dtype: int64
